## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-22-04-32-45 +0000,startribune,Live: Last place Canucks putting up a fight ag...,https://www.startribune.com/quinn-hughes-injur...
1,2026-11-22-04-28-43 +0000,nyt,China’s Spies Set Sights on A.I. Chips Years B...,https://www.nytimes.com/2026/11/21/world/asia/...
2,2026-11-22-04-17-24 +0000,startribune,Saturday rewind: Prep Bowl matchups set after ...,https://www.startribune.com/live-scores-update...
3,2026-11-22-04-05-27 +0000,nypost,LA-bound flight delayed after ICE officers mak...,https://nypost.com/2026/11/21/us-news/masked-i...
4,2026-11-22-03-44-19 +0000,nyt,Deadly Protests Erupt in Bangladesh Over Exile...,https://www.nytimes.com/2026/11/21/world/asia/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2756/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
63,trump,19
59,new,8
64,california,8
56,drug,7
165,fbi,7


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
178,2026-11-21-09-53-45 +0000,nypost,FBI’s ‘Most Wanted’ cartel kingpin ‘Chapo Isid...,https://nypost.com/2026/11/21/world-news/fbis-...,57
105,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...,54
113,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...,53
99,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...,50
109,2026-11-21-15-57-42 +0000,wapo,"Mexico seizes cartel boss ‘El Chapo Isidro,’ a...",https://www.washingtonpost.com/world/2026/11/2...,47


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
178,57,2026-11-21-09-53-45 +0000,nypost,FBI’s ‘Most Wanted’ cartel kingpin ‘Chapo Isid...,https://nypost.com/2026/11/21/world-news/fbis-...
99,50,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...
27,39,2026-11-22-00-17-14 +0000,nypost,Maniac accused of killing NYC woman in possibl...,https://nypost.com/2026/11/21/us-news/suspecte...
36,32,2026-11-21-23-50-12 +0000,chitri,Nations agree to extend talks on AI killer rob...,https://www.chicagotribune.com/2026/11/21/nati...
47,32,2026-11-21-22-36-00 +0000,nypost,Dramatic video shows Mexican drug kingpin ‘Cha...,https://nypost.com/2026/11/21/us-news/mexican-...
91,30,2026-11-21-17-38-58 +0000,nypost,Russian cargo ship bursts into flames in Balti...,https://nypost.com/2026/11/21/world-news/russi...
105,28,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...
145,26,2026-11-21-12-00-00 +0000,startribune,Do tech outages have you on edge? You’re not a...,https://www.startribune.com/do-tech-outages-ha...
44,23,2026-11-21-22-44-48 +0000,nypost,Grinning ex-NYPD cop admits to The Post that h...,https://nypost.com/2026/11/21/us-news/grinning...
94,23,2026-11-21-17-06-25 +0000,nypost,Former Italian police chief Gianni De Gennaro...,https://nypost.com/2026/11/21/world-news/forme...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
